# 福大教务处数据分析
## 1. 数据读取和格式设置

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["font.sans-serif"] = ["unifont"]
plt.rcParams["axes.unicode_minus"] = False

data_frame = pd.read_csv("notices.csv", encoding="utf-8-sig")
data_frame.head()

## 2. 统计各发布单位的通知数量占比

In [ ]:
publisher_counts = data_frame["发布单位"].value_counts()
publisher_ratio = (publisher_counts / len(data_frame) * 100).round(2)

fig, ax = plt.subplots(figsize=(8, 8))

ax.pie(
    publisher_counts,
    labels=publisher_counts.index, # type: ignore
    autopct="%1.1f%%",
    startangle=90,
    counterclock=False,
)

ax.set_title("发布单位数量占比", fontsize=14)
plt.tight_layout()
plt.show()

## 3. 课程调整类通知

In [ ]:
key_word = ["课程调整", "停课", "调课", "放假课程", "补上"]
pattern = "|".join(key_word)

title_mask = data_frame["标题"].str.contains(pattern, na=False)
body_mask = data_frame["正文"].str.contains(pattern, na=False)

course_adjust = data_frame[title_mask | body_mask].copy()
print("找到课程调整通知：", len(course_adjust), "条")

course_adjust.to_csv("course_adjust_notices.csv", index=False, encoding="utf-8-sig")

title_hits = []
body_hits = []
for kw in key_word:
    t = data_frame["标题"].str.contains(kw, na=False).sum()
    b = data_frame["正文"].str.contains(kw, na=False).sum()
    title_hits.append(t)
    body_hits.append(b)

x = range(len(key_word))
w = 0.38

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - w/2 for i in x], title_hits, w, label="标题命中", color="steelblue")
ax.bar([i + w/2 for i in x], body_hits, w, label="正文命中", color="coral")

ax.set_xticks(list(x))
ax.set_xticklabels(key_word, rotation=20)
ax.set_title("各关键词命中数", fontsize=13)
ax.set_ylabel("命中通知数")
ax.legend()

for i in x:
    ax.text(i - w/2, title_hits[i] + 0.1, str(title_hits[i]), ha="center", fontsize=9)
    ax.text(i + w/2, body_hits[i] + 0.1, str(body_hits[i]), ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 4. 附件下载次数与通知人的关系

In [ ]:
def parse_attachments(cell):
    if pd.isna(cell) or str(cell).strip() == "":
        return 0, 0
    n, total = 0, 0
    for part in str(cell).split(";"):
        part = part.strip()
        if not part:
            continue
        fields = part.split("|")
        n += 1
        
        try:
            total += int(fields[-1].strip())
        except (ValueError, IndexError):
            pass
    return n, total

data_frame[["附件数", "总下载次数"]] = data_frame["附件"].apply(
    lambda x: pd.Series(parse_attachments(x)))

attach_by_pub = data_frame.groupby("发布单位").agg(
    通知数=("标题", "count"),
    有附件通知数=("附件数", lambda s: (s > 0).sum()),
    附件总数=("附件数", "sum"),
    下载总次数=("总下载次数", "sum"),
    平均下载次数=("总下载次数", "mean")
).round(2)
attach_by_pub["平均每通知下载"] = (attach_by_pub["下载总次数"] / attach_by_pub["通知数"]).round(2)
attach_by_pub = attach_by_pub.sort_values("下载总次数", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

attach_by_pub["下载总次数"].plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("各通知人附件下载总次数", fontsize=13)
axes[0].set_ylabel("下载总次数")
axes[0].tick_params(axis="x", rotation=45)
for i, v in enumerate(attach_by_pub["下载总次数"]):
    axes[0].text(i, v + 50, str(int(v)), ha="center", fontsize=9)

attach_by_pub["平均每通知下载"].plot(kind="bar", ax=axes[1], color="coral")
axes[1].set_title("各通知人平均每通知附件下载次数", fontsize=13)
axes[1].set_ylabel("平均下载次数")
axes[1].tick_params(axis="x", rotation=45)
for i, v in enumerate(attach_by_pub["平均每通知下载"]):
    axes[1].text(i, v + 10, f"{v:.0f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 5. 通知发布的高峰期

In [ ]:
data_frame["发布时间"] = pd.to_datetime(data_frame["发布时间"], errors="coerce")
daily = data_frame.groupby(data_frame["发布时间"].dt.date).size().sort_index()

data_frame["周"] = data_frame["发布时间"].dt.to_period("W").dt.start_time
weekly = data_frame.groupby("周").size()

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

daily.plot(kind="line", marker="o", ax=axes[0], color="steelblue")
axes[0].set_title("每日通知发布数量", fontsize=13)
axes[0].set_ylabel("通知数")
axes[0].grid(alpha=0.3)

weekly.plot(kind="bar", ax=axes[1], color="coral")
axes[1].set_title("每周通知发布数量", fontsize=13)
axes[1].set_ylabel("通知数")
axes[1].tick_params(axis="x", rotation=45)
axes[1].grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## 6. 标题长度和附件下载次数相关系数

In [ ]:
data_frame["标题长度"] = data_frame["标题"].str.len()
data_frame["有附件"] = data_frame["附件数"] > 0

sub = data_frame[data_frame["有附件"]].copy()
corr = sub["标题长度"].corr(sub["总下载次数"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(sub["标题长度"], sub["总下载次数"], alpha=0.7, color="steelblue")
ax.set_xlabel("标题长度（字符）")
ax.set_ylabel("附件总下载次数")
ax.set_title(f"标题长度与附件下载次数（相关系数 {corr:.2f}）", fontsize=13)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()